In [ ]:
import os
import pandas as pd
import json
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

In [2]:
from typing import Any

def normalize_swiss_bill(value: Any) -> str | float:
    if pd.isna(value) or str(value).strip().lower() == "nan":
        return np.nan

    value_str = str(value).strip()

    if "." not in value_str:
        return value_str

    left, right = value_str.split(".", 1)
    return f"{left.zfill(2)}.{right.ljust(3, '0')}"


def business_number_to_affair_id(normalized_str: Any) -> str | float:
    if pd.isna(normalized_str) or str(normalized_str).strip().lower() == "nan":
        return np.nan

    left, right = str(normalized_str).split(".", 1)
    year_short = int(left)
    sequence = int(right)

    year_full = 2000 + year_short if year_short <= 26 else 1900 + year_short
    return f"{year_full}{str(sequence).zfill(4)}"

### Topic & Question

In [ ]:
TOPIC = "International relations"
QUESTION = "Should Switzerland strive for a free-trade agreement with the US?"
N_TOP_BILLS = 15

In [4]:
df = pd.read_csv("data/speeches_with_text.csv")

In [5]:
bill_types = ["Bill|Motion", "Bill|Parliamentary Initiative"]
df_filtered = df[df["Bill_Labels"].isin(bill_types)]
df_filtered = df_filtered.dropna(subset=["Bill_Number", "TEXT_SPAN"]).copy()
df_filtered = df_filtered.reset_index(drop=True)
df_filtered["Bill_Number"] = df_filtered["Bill_Number"].map(normalize_swiss_bill)
df_filtered["Affair_ID"] = df_filtered["Bill_Number"].map(business_number_to_affair_id)
df_filtered["Affair_ID"] = df_filtered["Affair_ID"].astype(str)

In [6]:
bills_metadata = json.load(open("data/bill_metadata_cache.json", "r"))

def get_bill_metadata(affair_id: str):
    return bills_metadata.get(affair_id, {})

In [7]:
get_bill_metadata("20060405")

{'Affair_ID': '20060405',
 'title': 'Doublement de la proportion de militaires en service long',
 'keywords': 'freiwilliger Wehrdienst, Soldat, junger Mensch',
 'ideological_summary': '<p>Les militaires en service long sont des soldats de milice qui accomplissent leur service militaire en une seule fois en 300 jours. Ils font ensuite partie de la réserve pendant dix ans et n\'accomplissent pas de cours de répétition. Cette forme de service est possible depuis 2004 dans presque tous les types de troupe.</p><p>Actuellement, la proportion maximale de militaires en service long est de 15 % du total des recrues. Nous proposons de la doubler et de l\'amener ainsi à un taux maximal de 30 %, ce qui correspond à un nombre maximal de l\'ordre de 6000 soldats. Cette action relativement simple a pour effet de permettre une importante modernisation de notre armée. Tout en demeurant fidèle au principe de milice, l\'armée suisse sera mieux à même de remplir les missions et de répondre aux menaces act

In [ ]:
embedder = SentenceTransformer("intfloat/multilingual-e5-large", device=torch.device("cuda" if torch.cuda.is_available() else "cpu"))

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4628.59it/s]


In [34]:
df_augmented = df_filtered.copy()
df_augmented["bill_metadata"] = df_augmented["Affair_ID"].map(get_bill_metadata, na_action="ignore")
df_augmented = df_augmented.reset_index(drop=True)
df_augmented = pd.concat([
    df_augmented.drop(columns=["bill_metadata", "Affair_ID"]),
    pd.json_normalize(df_augmented["bill_metadata"])
], axis=1)

In [35]:
question_embedding = embedder.encode(
    [f"query: {QUESTION}"],
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
question_embedding.shape

(1, 1024)

In [36]:
bill_meta = (
    df_augmented
    .groupby("Affair_ID", as_index=False)
    .agg(
        Bill_Number=("Bill_Number", "first"),
        Bill_Title_DE=("Bill_Title_DE", "first"),
        Bill_Title_FR=("Bill_Title_FR", "first"),
        title=("title", "first"),
        keywords=("keywords", "first"),
        ideological_summary=("ideological_summary", "first"),
        statutory_text=("statutory_text", "first"),
        n_paragraphs=("TEXT_SPAN", "size"),
        n_speeches=("Speech_ID", "nunique"),
    )
)

bill_meta["bill_search_text"] = bill_meta.apply(
    lambda row: " ".join([
        #str(row.get("Bill_Number", "")),
        str(row.get("title", "")),
        str(row.get("keywords", "")),
        str(row.get("ideological_summary", "")),
        #str(row.get("statutory_text", "")),
    ]).strip(),
    axis=1
)

In [37]:
bill_embeddings = embedder.encode(
    ["passage: " + x for x in bill_meta["bill_search_text"].fillna("").tolist()],
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

Batches: 100%|██████████| 21/21 [01:26<00:00,  4.10s/it]


In [38]:
scores = question_embedding @ bill_embeddings.T
bill_meta["bill_relevance_score"] = scores.max(axis=0)

In [39]:
candidate_bills = (
    bill_meta
    .sort_values("bill_relevance_score", ascending=False)
    .head(N_TOP_BILLS)
)

In [40]:
df_target_speeches = df_augmented[df_augmented["Affair_ID"].astype(str).isin(set(candidate_bills["Affair_ID"].astype(str)))].copy()

In [41]:
passage_embeddings = embedder.encode(
    ["passage: " + x for x in df_target_speeches["TEXT_SPAN"].fillna("").tolist()],
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
passage_embeddings.shape

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches: 100%|██████████| 2/2 [00:04<00:00,  2.14s/it]


(455, 1024)

In [42]:
question_scores = (question_embedding @ passage_embeddings.T).flatten()

In [43]:
passages = df_target_speeches.copy()
passages["embedding_score"] = question_scores

In [44]:
# High-recall candidate pool
candidate_passages = (
    passages
    .sort_values("embedding_score", ascending=False)
    .copy()
)

In [20]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4016.48it/s]


In [45]:
candidate_passages["rerank_score"] = reranker.predict(
    [[QUESTION, text] for text in candidate_passages["TEXT_SPAN"].fillna("")],
    batch_size=256,
    show_progress_bar=True,
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches: 100%|██████████| 2/2 [00:03<00:00,  1.78s/it]


In [46]:
RERANK_THRESHOLD = 0.5
top_evidence = candidate_passages[candidate_passages["rerank_score"] > RERANK_THRESHOLD].copy()
top_evidence = top_evidence.sort_values("rerank_score", ascending=False)

In [28]:
from transformers import pipeline

classifier = pipeline(
    task="text-classification",
    model="MatteoFasulo/xlm-roberta-xstance"
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4370.00it/s]


In [47]:
HYPOTHESIS = "Switzerland should strive for a free-trade agreement with the US"
stance_results = classifier(
    [
        {
            "text": HYPOTHESIS,
            "text_pair": text,
        }
        for text in top_evidence["TEXT_SPAN"].fillna("").tolist()
    ],
    batch_size=256,
    show_progress_bar=True,
)

In [48]:
df_stances = pd.concat([
    top_evidence.reset_index(drop=True),
    pd.DataFrame(stance_results)
], axis=1)
df_stances = df_stances.rename(columns={"score": "stance_score", "speaker_fullname": "MP", "language_of_text": "language", "TEXT_SPAN": "passage"})

In [49]:
df_stances['question'] = QUESTION
df_stances['topic'] = TOPIC
df_stances[
    [
        "Speech_ID",
        "Paragraph_ID",
        "Affair_ID",
        "Bill_Number",
        "language",
        "question",
        "passage",
        "label",
        "stance_score",
        "MP",
        "party",
        "topic",
        "rerank_score",
    ]
].to_dict('records')

[{'Speech_ID': 248779,
  'Paragraph_ID': 5,
  'Affair_ID': '20183797',
  'Bill_Number': '18.3797',
  'language': 'FR',
  'question': 'Should Switzerland strive for a free-trade agreement with the US?',
  'passage': "Pour la Suisse, un accord de libre-échange avec les Etats-Unis offrirait de nouvelles opportunités pour l'économie, cela garantirait de la sécurité juridique, et cela créerait surtout un cadre institutionnalisé pour renforcer les relations économiques bilatérales. Naturellement, cela implique certains défis, en particulier pour le secteur agricole, dans lequel les Etats-Unis poursuivent des intérêts offensifs. Il convient donc de clarifier dans quelle mesure nous pourrions avoir une certaine flexibilité, où nous pourrions avoir des solutions pour des produits sensibles, quel degré de libéralisation acceptable nous pourrions trouver pour les deux pays. Il faut noter aussi - et cela a été relevé par le rapporteur - que les spécialités de fromage sont de plus en plus prisées a

In [50]:
df_stances[
    [
        "Speech_ID",
        "Paragraph_ID",
        "Affair_ID",
        "Bill_Number",
        "language",
        "question",
        "passage",
        "label",
        "stance_score",
        "MP",
        "party",
        "topic",
        "rerank_score",
    ]
].to_json(f"{TOPIC.replace(' ', '_').lower()}_{QUESTION.replace(' ', '_').lower()}.json", orient='records', force_ascii=False, indent=2)